In [2]:
%matplotlib inline

import os
import sys
import math
import time
import json
import requests

from sys import platform
from pathlib import Path
from functools import partial

sys.path.append('../../')

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from importlib.metadata import version 

import tiktoken
import numpy as np
import tensorflow

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

%load_ext autoreload
%autoreload 2

from llm_from_scratch.CH4.gpt import GPTModel, generate_text_simple
from llm_from_scratch.CH5.utils import load_weights_into_gpt, text_to_token_ids, token_ids_to_text, generate, plot_losses
from llm_from_scratch.CH5.gpt_download import download_and_load_gpt2
from llm_from_scratch.CH5.loss import calc_loss_batch, calc_loss_loader
from llm_from_scratch.CH5.optim import train_model

# from spam_dataset import SpamDataset, download_and_unzip_spam_data, create_balanced_dataset, random_split
# from utils import calc_accuracy_loader, calc_loss_loader, calc_loss_batch, train_classifier
from llm_from_scratch.CH6.optim import create_optimizer, LayerDecayValueAssigner

from utils import check_if_running, query_model, generate_model_scores
from instruction_data import download_and_load_file, format_input, InstructionDataset, custom_collate_fn

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
result=query_model('What do Llamas eat?')
print(result)

Llamas are herbivores, which means they primarily feed on plant-based foods. Their diet typically consists of:

1. Grasses: Llamas love to graze on various types of grasses, including tall grasses, short grasses, and even weeds.
2. Hay: High-quality hay, such as alfalfa or timothy hay, is a staple in a llama's diet. They enjoy the sweet taste and texture of fresh hay.
3. Grains: Llamas may receive grains like oats, barley, or corn as part of their daily ration. However, it's essential to provide these grains in moderation, as they can be high in calories.
4. Fruits and vegetables: Llamas enjoy a variety of fruits and veggies, such as apples, carrots, sweet potatoes, and leafy greens like kale or spinach.
5. Minerals: Llamas require access to mineral supplements, which help maintain their overall health and well-being.

In the wild, llamas might also eat:

1. Leaves: They'll munch on leaves from trees and shrubs, including plants like willow, alder, and birch.
2. Bark: In some cases, ll

In [4]:
def extract_instruction(text):
    for content in text.split("\n"):
        if content: return content.strip()

query="<|begin_of_text|><|start_header_id|>user<|end_header_id|>"
result=query_model(query, role='assistant')
print(f"{result=}")
instruction=extract_instruction(result)
print(f"{instruction=}")

result="\n\nI'm trying to create a character for a story and I want her to have a unique name. I've been brainstorming, but nothing seems quite right. Can you help me come up with some ideas?\n\nWhat kind of vibe are you going to give me?"
instruction="I'm trying to create a character for a story and I want her to have a unique name. I've been brainstorming, but nothing seems quite right. Can you help me come up with some ideas?"


In [5]:
response=query_model(instruction, role='user')
print(response)

I'd be happy to help! Creating a unique and fitting name for your character can be a fun and creative process. Let's get started!

To get the ball rolling, can you give me a sense of what kind of story this character is in? Is it:

1. Fantasy (e.g., medieval setting, magic, mythical creatures)?
2. Science fiction (e.g., futuristic world, space travel, advanced technology)?
3. Contemporary (e.g., realistic, everyday life, no supernatural elements)?
4. Historical (e.g., set in a specific time period, cultural or social context)?

Also, what kind of personality or traits do you envision for your character? For example:

1. Strong and confident?
2. Quiet and introverted?
3. Adventurous and bold?
4. Creative and artistic?

Lastly, are there any specific themes, settings, or cultures that might inspire the name? (e.g., nature-inspired names, cultural references, mythological allusions)

Feel free to share as much or as little as you'd like, and I'll start generating some ideas for you!
